#### Configuração do ambiente

- Crie o ambiente virtual python

```bash
python3 -m venv venv
```

- Ative o ambiente

```bash
source venv/bin/activate
```

- Instale as dependências

```bash
pip install -r requirements_nlp.txt
```

#### Tokenização

In [1]:
import pandas as pd
import numpy as np
import nltk

In [2]:
nltk.download('punkt') # Modelo pré treinado de tokenização do NLTK
nltk.download('punkt_tab') # Auxiliar do punkt para outras línguas

[nltk_data] Downloading package punkt to /home/codespace/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [3]:
from nltk.tokenize import word_tokenize

In [4]:
## Usando o tokenizador do NLTK em uma frase
frase = "Hobbit não é uma medida de informação"
print(word_tokenize(frase))

['Hobbit', 'não', 'é', 'uma', 'medida', 'de', 'informação']


In [5]:
## Simulando um Dataset de reviews de filmes
data = {
    "Comentario": ["Simplesmente incrível! A fotografia é deslumbrante, a trilha sonora é envolvente e os atores entregaram performances impecáveis. Recomendo para quem gosta de uma história bem construída e emocionante. Nota 10/10!", 
              "Não gostei nada do filme. A trama é confusa, os personagens não têm profundidade e o final foi totalmente decepcionante.",
              "O filme é mediano. Tem alguns momentos interessantes e boas atuações, mas também possui partes arrastadas e previsíveis"],
    "Estrelas": [5,1,3]    
}

df = pd.DataFrame(data)
df.iloc[0]

Comentario    Simplesmente incrível! A fotografia é deslumbr...
Estrelas                                                      5
Name: 0, dtype: object

In [6]:
## Tokenizando todos os comentários
df["Tokens"] = df["Comentario"].apply(lambda x: word_tokenize(x))

In [7]:
## Descobrindo onde existem termos de usuários promotores
palavras_chave_promotores = ["adorei", "incrível", "ótimo", "recomendo"]
df["Termo_Promotor"] = df["Tokens"].apply(lambda x: 1 if set(x) & set(palavras_chave_promotores) else 0)

In [8]:
df

,Comentario,Estrelas,Tokens,Termo_Promotor
0,Simplesmente incrível! A fotografia é deslumbr...,5,"[Simplesmente, incrível, !, A, fotografia, é, ...",1
1,"Não gostei nada do filme. A trama é confusa, o...",1,"[Não, gostei, nada, do, filme, ., A, trama, é,...",0
2,O filme é mediano. Tem alguns momentos interes...,3,"[O, filme, é, mediano, ., Tem, alguns, momento...",0


Com esses tokens podemos investigar o relacionamento entre os termos usados e a avaliação (Estrelas), trazendo insights sobre cada produto.

A tokenização utilizando o package NLTK é bastante comum para pré-processamento de texto. Para uso em modelos de IA, é mais comum a tokenização com "subwords". O package "transformers" do Hugginface oferece essa funcionalidade.

```bash
pip install transformers
```

In [9]:
from transformers import AutoTokenizer

/workspaces/ml-supervised-dev/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
# Modelo baseado em português
tokenizer = AutoTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")

In [11]:
texto = "Um exemplo de frase para tokenizar usando o transformers do huggingface"
tokens = tokenizer.tokenize(texto)
print(tokens)

['Um', 'exemplo', 'de', 'frase', 'para', 'to', '##ken', '##izar', 'usando', 'o', 'transform', '##ers', 'do', 'h', '##ug', '##gin', '##g', '##face']


---

#### Stopwords

In [12]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords') # Faz o download das stopwords existente no NLTK

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [13]:
texto = "Um exemplo de frase para removermos as stopwods"
tokens = word_tokenize(texto)

In [14]:
stopwords_pt = set(stopwords.words('portuguese')) # Stopwords em português
# É possível adicionar mais stopwords a esse set, ou seja, 
# fazer seu conjunto personalizado de stopwords

In [15]:
palavras_sem_stopwords = [p for p in tokens if p.lower() not in stopwords_pt]
print(palavras_sem_stopwords)

['exemplo', 'frase', 'removermos', 'stopwods']


---

#### Stemming e Lemmatization

In [16]:
import nltk
from nltk.stem import RSLPStemmer
nltk.download('rslp')  # Para português

stemmer = RSLPStemmer()

[nltk_data] Downloading package rslp to /home/codespace/nltk_data...
[nltk_data]   Unzipping stemmers/rslp.zip.


In [17]:
cantar = ["canto", "cantas", "canta", "cantamos", "cantais", "cantam", "cantaram", "cantavam"]
for palavra in cantar:
    print(f"Palavra: {palavra} | Stem: {stemmer.stem(palavra)}")

Palavra: canto | Stem: cant
Palavra: cantas | Stem: cant
Palavra: canta | Stem: cant
Palavra: cantamos | Stem: cant
Palavra: cantais | Stem: cant
Palavra: cantam | Stem: cant
Palavra: cantaram | Stem: cant
Palavra: cantavam | Stem: cant


In [18]:
import spacy
spacy.cli.download("pt_core_news_sm") # Modelo para português
nlp = spacy.load('pt_core_news_sm')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 21.3 MB/s eta 0:00:0000:0100:01



[notice] A new release of pip is available: 23.2.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [19]:
texto = "Estamos cantando hoje"

In [20]:
doc = nlp(texto)
# Lemmatização
lemmas = [token.lemma_ for token in doc]
print(lemmas)

['estar', 'cantar', 'hoje']


---